<a href="https://colab.research.google.com/github/hongxuzhou/undo_flip_flop/blob/main/undo_flip_flop_mamba2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#[UNDO] Flip Flop vs. Standard Flip Flop: Mamba 2

## 0. Environment Setup

In [ ]:
!pip install causal-conv1d>=1.4.0 --no-build-isolation
!pip install mamba-ssm --no-build-isolation

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.7/121.7 kB 4.0 MB/s eta 0:00:00
  Preparing metadata (pyproject.toml) ... done
  Created wheel for mamba-ssm: filename=mamba_ssm-2.3.1-cp312-cp312-linux_x86_64.whl size=533592144 sha256=d66a3c4c94a693e02d341cace7a6af0b72177b6afa655a25e3a6505130a68cbf
  Stored in directory: /root/.cache/pip/wheels/28/83/54/d45107838fec575b93f5d723f56351cee19a1b13bcd4ec9f3f
Successfully built mamba-ssm


In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

## 1. Task Definition
This notebook investigates whether Mamba-2, a structured state space model (SSM),
can learn **stack-based rollback**: a form of memory that requires tracking
not just the current state, but the history of how that state was reached.

We compare two tasks built on the Flip-Flop language modelling framework

---

**Standard Flip-Flop.** A sequence of write (`w`), read (`r`), and value tokens
(`0`, `1`). The model must remember the most recently written value and reproduce
it at every `r`.
This is a regular language task. Prior work has shown that a single-layer Mamba-2
can solve it.

---

**UNDO Flip-Flop.** An `u` (undo) token is added. It rolls back the register
to the value it held *before* the most recent write — a last-in, first-out (LIFO)
operation. Arbitrarily nested undos require maintaining a write history, not just
a single register value.

After `u`, the register reverts to `0` (the value written before `w 1`).
This places the task in bounded context-free territory: tracking depth-bounded
stack dynamics that a finite SSM state must somehow encode.

---

The central question is not whether Mamba-2 *can* express this computation
in principle (theoretical analysis suggests a 2-layer model has sufficient
capacity (Sarrof et al., 2024)) but whether gradient-based optimisation reliably *learns* it.

In [ ]:

# Vocabulary and constants
VOCAB = {'0': 0, '1': 1, 'r': 3, 'i': 4, 'w': 5, 'u': 6}
READ_TOKEN = VOCAB['r']
IGNORE_INDEX = -100

# The pad token for inputs cannot be -100, as this would cause an error in the Embedding layer.
# We use 'i' (ignore) as the input padding token, since its labels are masked out anyway.
PAD_TOKEN_INPUT = VOCAB['i']

class FlipFlopDataset(Dataset):
    def __init__(self, file_path):
        # Load the .pt dataset file
        self.data = torch.load(file_path)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        # Extract the pre-generated input and label tensors
        # Use clone().detach() for safe GPU transfer and cast to long tensor
        input_seq = self.data[idx]['input'].clone().detach().long()
        label_seq = self.data[idx]['label'].clone().detach().long()
        return input_seq, label_seq

def collate_fn(batch):
    # Use clone().detach() for safe GPU transfer and cast to long tensor
    inputs = [item[0] for item in batch]
    # Extract the second element of each tuple (label_seq)
    labels = [item[1] for item in batch]

    # Pad sequences
    # Pad inputs with 4 ('i'), labels with -100
    inputs_padded = pad_sequence(inputs, batch_first=True, padding_value=PAD_TOKEN_INPUT)
    labels_padded = pad_sequence(labels, batch_first=True, padding_value=IGNORE_INDEX)

    return {
        'input_ids': inputs_padded.cuda(),
        'labels': labels_padded.cuda()
    }

#--- Dataset instantiation ---


# Standard Flip-Flop (Task A)
std_train_id_dataset = FlipFlopDataset("/content/drive/MyDrive/02_EM-LCT/uds_course/TAL/standard_lff_dataset/std_train_id_ff.pt")
std_train_id_loader = DataLoader(std_train_id_dataset, batch_size=16, shuffle=True, collate_fn=collate_fn)

std_test_id_dataset = FlipFlopDataset("/content/drive/MyDrive/02_EM-LCT/uds_course/TAL/standard_lff_dataset/std_test_id_ff.pt")
std_test_id_loader = DataLoader(std_test_id_dataset, batch_size=16, shuffle=False, collate_fn=collate_fn)

std_test_ood_dataset = FlipFlopDataset("/content/drive/MyDrive/02_EM-LCT/uds_course/TAL/standard_lff_dataset/std_test_ood_ff.pt")
std_test_ood_loader = DataLoader(std_test_ood_dataset, batch_size=16, shuffle=False, collate_fn=collate_fn)

# UNDO Flip-Flop (Task B)
undo_train_id_dataset = FlipFlopDataset("/content/drive/MyDrive/02_EM-LCT/uds_course/TAL/undo_dataset/undo_train_id.pt")
undo_train_id_loader = DataLoader(undo_train_id_dataset, batch_size=16, shuffle=True, collate_fn=collate_fn)

undo_test_id_dataset = FlipFlopDataset("/content/drive/MyDrive/02_EM-LCT/uds_course/TAL/undo_dataset/undo_test_id.pt")
undo_test_id_loader = DataLoader(undo_test_id_dataset, batch_size=16, shuffle=False, collate_fn=collate_fn)

undo_test_ood_dataset = FlipFlopDataset("/content/drive/MyDrive/02_EM-LCT/uds_course/TAL/undo_dataset/undo_test_ood.pt")
undo_test_ood_loader = DataLoader(undo_test_ood_dataset, batch_size=16, shuffle=False, collate_fn=collate_fn)

## 2. Model Architecture
`Mamba2ForFlipFlop` is a minimal sequence model: a token embedding, one or more
Mamba-2 blocks, and a linear output head projecting back to the vocabulary.

input_ids -> Embedding -> [Mamba2 * n_layers] -> Linear -> Logits

All experiments use `d_model=16`, `d_state=16`, following the small-model regime
in Sarrof et al. (2024). This is intentional: the question is about what the
architecture *can learn*, not whether a larger model can brute-force the task.

We test two configurations:

- **1-layer**: theoretically insufficient for stack-depth tracking. Expected to
  fail on UNDO FF regardless of training duration.
- **2-layer**: theoretically expressive enough. Whether it *learns* the correct
  mechanism is the empirical question this notebook addresses.

In [ ]:
import torch.nn as nn
from mamba_ssm import Mamba2

class Mamba2ForFlipFlop(nn.Module):
    def __init__(self, vocab_size=7, d_model=16, d_state=16, d_conv=4, expand=2, n_layers=1):
        super().__init__()

        # 1. Token embedding layer (maps indices 0–6 to dense vectors of dimension d_model)
        self.embedding = nn.Embedding(vocab_size, d_model)

        # 2. Dynamically compute a safe headdim to avoid Mamba2 assertion errors
        d_inner = d_model * expand
        safe_headdim = 16 if d_inner % 16 == 0 else 8

        # 3. Stack n_layers Mamba2 blocks
        self.layers = nn.ModuleList([
            Mamba2(
                d_model=d_model,
                d_state=d_state,
                d_conv=d_conv,
                expand=expand,
                headdim=safe_headdim  # Explicitly pass a safe headdim
            ) for _ in range(n_layers)
        ])

        # 4. Output projection head mapping d_model back to vocabulary logits
        self.lm_head = nn.Linear(d_model, vocab_size)

    def forward(self, input_ids):
        # input_ids shape: (batch_size, seq_length)

        # Embed input tokens: (batch_size, seq_length, d_model)
        x = self.embedding(input_ids)

        # Pass through all Mamba2 layers sequentially
        for layer in self.layers:
            x = layer(x)

        # Project to vocabulary space: (batch_size, seq_length, vocab_size)
        logits = self.lm_head(x)
        return logits

# ==========================================
#========== Instantiate models ==========
# ==========================================
# Hypothesis 1: the 1-layer model lacks the representational capacity for stack-depth tracking and should fail to converge on UNDO FF
model_1_layer = Mamba2ForFlipFlop(n_layers=1).cuda()

# Hypothesis 2: the 2-layer model can learn bounded hierarchical rollback on UNDO FF, potentially via a stack-free shortcut
model_2_layer = Mamba2ForFlipFlop(n_layers=2).cuda()

## 3. Main Experiment: Standard FF vs. UNDO FF
We train four models for up to 100 epochs
with early stopping, including 1-layer and 2-layer on each task.

**Evaluation** uses sequence-level exact match: a sequence is counted as correct
only if *every* read token in it is predicted correctly. This is stricter than
token-level accuracy and rules out partial credit from lucky guesses.

**Early stopping** triggers when ID exact-match accuracy reaches 100% or loss
drops below 1e-4. This avoids wasting compute once convergence is confirmed, and
makes the *epoch at which convergence occurs* a meaningful signal in itself.

**Hyperparameters** follow Sarrof et al. (2024): AdamW, lr=3e-4. Training and
test sequences are drawn from lengths 1–50 (in-distribution, ID). A separate
OOD split covering lengths 51–100 probes length generalisation.

In [ ]:
from torch.optim import AdamW

# ==========================================
# 1. Sequence-level exact-match evaluation
# ==========================================
def evaluate(model, dataloader):
    model.eval()
    correct_sequences = 0
    total_sequences = 0

    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch['input_ids']
            labels = batch['labels']

            # Forward pass
            logits = model(input_ids)
            preds = torch.argmax(logits, dim=-1)

            # Identify valid prediction positions (i.e., where label != -100)
            valid_mask = (labels != -100)

            # Compare predictions against ground-truth labels
            correct_preds = (preds == labels)

            # Iterate over sequences in the batch
            for i in range(len(labels)):
                seq_mask = valid_mask[i]
                if seq_mask.sum() > 0: # Ensure the sequence contains at least one read token to predict
                    # Sequence-level exact match: all valid predictions in the sequence must be correct
                    if (correct_preds[i][seq_mask]).sum() == seq_mask.sum():
                        correct_sequences += 1
                total_sequences += 1

    # Return accuracy
    return correct_sequences / total_sequences if total_sequences > 0 else 0

# ==========================================
# 2. Unified training loop
# ==========================================
def run_experiment(model, train_loader, test_id_loader, test_ood_loader, task_name, num_epochs=100):
    # Following Sarrof et al. hyperparameters: AdamW, lr=3e-4
    optimizer = AdamW(model.parameters(), lr=3e-4)
    criterion = nn.CrossEntropyLoss(ignore_index=-100)

    print(f"\nStarting training: {task_name}")

    for epoch in range(num_epochs):
        model.train()
        total_loss = 0

        for batch in train_loader:
            optimizer.zero_grad()

            input_ids = batch['input_ids']
            labels = batch['labels']

            logits = model(input_ids)

            # Flatten for CrossEntropyLoss
            loss = criterion(logits.view(-1, logits.size(-1)), labels.view(-1))

            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)

        # --- Early stopping ---
        # After each epoch, check convergence on the in-distribution (ID) test set
        id_acc = evaluate(model, test_id_loader)
        print(f"   Epoch {epoch+1:03d}/{num_epochs} - Loss: {avg_loss:.4f} | ID acc: {id_acc:.2%}")

        # Early stopping condition: 100% sequence-level accuracy or loss < 1e-4
        if id_acc == 1.0 or avg_loss < 1e-4:
            print(f"Early stop triggered at epoch {epoch+1}: model has fully converged.")
            break

    print(f"\\nFinal evaluation:{task_name}")

    # Final evaluation — ID accuracy should be 100% if early stopping was triggered
    final_id_acc = evaluate(model, test_id_loader)
    final_ood_acc = evaluate(model, test_ood_loader)

    print(f" ID  (length 1–50): {final_id_acc:.2%}")
    print(f" OOD (length 51–100):: {final_ood_acc:.2%}")
    print("=" * 60)

    return final_id_acc, final_ood_acc

# ==========================================
# 3. Run comparison experiments (execution block)
# ==========================================
# Training stops as soon as convergence is detected, regardless of num_epochs

print("=== Group 1: Standard Flip-Flop (Task A) baseline ===")
std_model_1L = Mamba2ForFlipFlop(n_layers=1).cuda()
std_model_2L = Mamba2ForFlipFlop(n_layers=2).cuda()

run_experiment(std_model_1L, std_train_id_loader, std_test_id_loader, std_test_ood_loader, "Standard FF (1-Layer)", num_epochs=100)
run_experiment(std_model_2L, std_train_id_loader, std_test_id_loader, std_test_ood_loader, "Standard FF (2-Layer)", num_epochs=100)

print("\\n=== Group 2: UNDO Flip-Flop (Task B) ===")
undo_model_1L = Mamba2ForFlipFlop(n_layers=1).cuda()
undo_model_2L = Mamba2ForFlipFlop(n_layers=2).cuda()

run_experiment(undo_model_1L, undo_train_id_loader, undo_test_id_loader, undo_test_ood_loader, "UNDO FF (1-Layer)", num_epochs=100)
run_experiment(undo_model_2L, undo_train_id_loader, undo_test_id_loader, undo_test_ood_loader, "UNDO FF (2-Layer)", num_epochs=100)



=== Group 1: Standard Flip-Flop (Task A) baseline ===

Starting training: Standard FF (1-Layer)
   Epoch 001/100 - Loss: 0.7841 | ID acc: 49.20%
   Epoch 002/100 - Loss: 0.4525 | ID acc: 81.00%
   Epoch 003/100 - Loss: 0.1250 | ID acc: 96.30%
   Epoch 004/100 - Loss: 0.0385 | ID acc: 98.80%
   Epoch 005/100 - Loss: 0.0108 | ID acc: 99.85%
   Epoch 006/100 - Loss: 0.0037 | ID acc: 99.95%
   Epoch 007/100 - Loss: 0.0020 | ID acc: 99.95%
   Epoch 008/100 - Loss: 0.0013 | ID acc: 99.95%
   Epoch 009/100 - Loss: 0.0006 | ID acc: 100.00%
Early stop triggered at epoch 9: model has fully converged.
\nFinal evaluation:Standard FF (1-Layer)
 ID  (length 1–50): 100.00%
 OOD (length 51–100):: 98.55%

Starting training: Standard FF (2-Layer)
   Epoch 001/100 - Loss: 0.6439 | ID acc: 90.45%
   Epoch 002/100 - Loss: 0.0555 | ID acc: 99.30%
   Epoch 003/100 - Loss: 0.0103 | ID acc: 99.80%
   Epoch 004/100 - Loss: 0.0034 | ID acc: 99.85%
   Epoch 005/100 - Loss: 0.0011 | ID acc: 99.90%
   Epoch 006/100

(0.9825, 0.893)

## 4. Stress Test: Aggressive UNDO Pressure
The 2-layer model achieves high ID accuracy. This section asks whether that
accuracy reflects genuine stack tracking or a superficial heuristic.

The aggressive test set holds sequence length fixed at 50, within the ID range, while packs in 10 consecutive `u`
operations. This forces the model to maintain and correctly unwind a deep write
history within a single sequence.

A model that has learned proper LIFO rollback should handle this without
degradation. A model relying on a shortcut will break under the accumulated
retraction pressure.

In [ ]:
# 1. Instantiate the DataLoader for the aggressive UNDO pressure test
# Replace with the actual path to your dataset file !!!
aggressive_dataset = FlipFlopDataset("/content/drive/MyDrive/02_EM-LCT/uds_course/TAL/undo_dataset/aggressive_d10.pt")
aggressive_loader = DataLoader(aggressive_dataset, batch_size=16, shuffle=False, collate_fn=collate_fn)

# 2. Evaluate the converged undo_model_2L (already in GPU memory)
print("\\n=== Aggressive UNDO Pressure Test ===")
print("Conditions: sequence length fixed at 50 (within ID range), containing 10 consecutive dense stack-pop rollbacks.")

# Note: inference only — no gradient computation, no weight updates
aggressive_acc = evaluate(undo_model_2L, aggressive_loader)

print(f"   Baseline ID accuracy (length 1–50): 98.60%")
print(f"   Aggressive UNDO (10 consecutive rollbacks):  {aggressive_acc:.2%}")
print("=" * 60)

\n=== Aggressive UNDO Pressure Test ===
Conditions: sequence length fixed at 50 (within ID range), containing 10 consecutive dense stack-pop rollbacks.
   Baseline ID accuracy (length 1–50): 98.60%
   Aggressive UNDO (10 consecutive rollbacks):  47.30%


## 5. Mechanistic Analysis: Black-Box Causal Ablation
High accuracy alone does not distinguish between two very different strategies:

- **Genuine stack rollback**: the model retrieves the correct deep history value
  after each `u`, maintaining a proper LIFO register.
- **Local toggle heuristic**: the model treats `u` as "flip the most recently
  seen value" — a shortcut that works in simple cases but is not stack behaviour.

To separate these, we apply two targeted perturbations to sequences the model
already predicts correctly.

For each such sequence, we simulate the stack to identify two positions:

- **v1** — the current stack top: the value that `r` *should* read.
- **v2** — the most recently popped value: already discarded by `u`, so it
  *should not* affect the output.

Stack after u: [0]   ← v1 = 0 (at write position of '0')
Popped:        [1]   ← v2 = 1 (at write position of '1')
Correct output at r: 0

**Perturbation A — Deep History Ablation.** Flip v1 and re-run the model.
A genuine stack-based system must change its output. If the output is unchanged,
the model never retrieved the deep history.

**Perturbation B — Surface Distractor Ablation.** Flip v2 and re-run the model.
A genuine stack-based system must be unaffected. If the output changes, the model
is sensitive to a value it should have discarded — evidence of a local toggle
heuristic rather than a proper pop.

The two rates reported are:

- **Deep History Loss Rate**: proportion of UNDO operations where Perturbation A
  produced no output change.
- **Local Toggle Heuristic Rate**: proportion of UNDO operations where
  Perturbation B produced an output change.

In [ ]:
import copy

def run_causal_ablation_test(model, dataloader):
    model.eval()

    valid_samples = 0
    deep_history_loss_count = 0      # Count Perturbation A failures: deep history changed but output unchanged (model did not retrieve it)
    surface_distractor_flip_count = 0  # Count Perturbation B failures: popped surface value changed and output followed (local toggle heuristic)

    print("Running black-box causal ablation scan...")

    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch['input_ids']
            labels = batch['labels']

            # Obtain baseline predictions
            logits = model(input_ids)
            preds = torch.argmax(logits, dim=-1)

            # Mask of valid prediction positions (read token locations)
            read_mask = (labels != -100)

            for i in range(len(input_ids)):
                seq = input_ids[i]
                label = labels[i]
                pred = preds[i]

                seq_mask = read_mask[i]
                # Only analyse sequences the model predicted correctly (to isolate the heuristic)
                if seq_mask.sum() == 0 or (pred[seq_mask] != label[seq_mask]).any():
                    continue

                # ==========================================
                # 1. Simulate the stack to locate v1 (deep history) and v2 (surface distractor) indices
                # ==========================================
                seq_list = seq.tolist()
                stack = []   # Indices of written values
                popped = []  # Indices of popped values

                for idx, token in enumerate(seq_list):
                    if token == 5: # 'w' (write)
                        stack.append(idx + 1) # The token immediately after the "w" is the written value, record its index
                    elif token == 6: # 'u' (undo)
                        if len(stack) > 1:
                            popped.append(stack.pop())

                # Skip seqnences with no valid pop or an empty stack
                if not popped or not stack:
                    continue

                deep_idx = stack[-1]     # v1：current stack top — the value that should be read by "r"
                surface_idx = popped[-1] # v2：most recently popped value — should not influence the final output

                valid_samples += 1

                # ==========================================
                # 2. Perturbation A: Deep History Ablation
                # ==========================================
                # Flip v1 (0→1 or 1→0). A genuine stack-based system must change its output accordingly.
                seq_A = seq.clone()
                seq_A[deep_idx] = 1 - seq_A[deep_idx]

                logits_A = model(seq_A.unsqueeze(0))
                pred_A = torch.argmax(logits_A, dim=-1).squeeze(0)

                # If the output does not change, the model failed to retrieve the deep history
                if (pred_A[seq_mask] == pred[seq_mask]).all():
                    deep_history_loss_count += 1

                # ==========================================
                # 3. Perturbation B: Surface Distractor Ablation
                # ==========================================
                # Flip v2 while keeping v1 unchanged. A genuine stack-based system must be unaffected.
                seq_B = seq.clone()
                seq_B[surface_idx] = 1 - seq_B[surface_idx]

                logits_B = model(seq_B.unsqueeze(0))
                pred_B = torch.argmax(logits_B, dim=-1).squeeze(0)

                # If the output changes, the model is applying a local toggle heuristic rather than a proper pop
                if (pred_B[seq_mask] != pred[seq_mask]).any():
                    surface_distractor_flip_count += 1

    print("\n" + "=" * 60)
    print(f"Causal ablation results ({valid_samples} correctly predicted sequences analysed)")
    print("-" * 60)

    if valid_samples > 0:
        deep_loss_rate = deep_history_loss_count / valid_samples
        surface_flip_rate = surface_distractor_flip_count / valid_samples
        print(f"Deep History Loss Rate: {deep_loss_rate:.2%}")
        print(f"  (Proportion of UNDO operations where the model failed to retrieve the correct deep history)")

        print(f"Local Toggle Heuristic Rate:{surface_flip_rate:.2%}")
        print(f"   (Proportion of UNDO operations where the model toggled the most recent value rather than performing a genuine pop)")
    else:
        print("No valid correctly-predicted sequences containing UNDO operations were found.")
    print("=" * 60)

# Run the ablation on the ID test set using the converged 2-layer model
run_causal_ablation_test(undo_model_2L, undo_test_id_loader)

Running black-box causal ablation scan...

Causal ablation results (545 correctly predicted sequences analysed)
------------------------------------------------------------
Deep History Loss Rate: 2.75%
  (Proportion of UNDO operations where the model failed to retrieve the correct deep history)
Local Toggle Heuristic Rate:37.43%
   (Proportion of UNDO operations where the model toggled the most recent value rather than performing a genuine pop)
